In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set project directory
project_path = Path.cwd()

# Processed data directory
processed_path = project_path / "data" / "processed"

# Load processed datasets
baggage = pd.read_csv(processed_path / "baggage_clean.csv")
flight = pd.read_csv(processed_path / "flight_clean.csv")
baggage_assignment = pd.read_csv(
    processed_path / "baggage_assignment_clean.csv"
)
baggage_scan_event = pd.read_csv(
    processed_path / "baggage_scan_event_clean.csv"
)

# Convert timestamp columns back to UTC-aware datetime
baggage["created_at"] = pd.to_datetime(
    baggage["created_at"], utc=True
)

flight["departure_time"] = pd.to_datetime(
    flight["departure_time"], utc=True
)
flight["arrival_time"] = pd.to_datetime(
    flight["arrival_time"], utc=True
)

baggage_assignment["assigned_at"] = pd.to_datetime(
    baggage_assignment["assigned_at"], utc=True
)

baggage_scan_event["event_time"] = pd.to_datetime(
    baggage_scan_event["event_time"], utc=True
)
baggage_scan_event["received_at"] = pd.to_datetime(
    baggage_scan_event["received_at"], utc=True
)

print("All processed datasets loaded successfully!")

print("\nDataset dimensions:")
print("Baggage:", baggage.shape)
print("Flight:", flight.shape)
print("Assignment:", baggage_assignment.shape)
print("Scan Events:", baggage_scan_event.shape)

All processed datasets loaded successfully!

Dataset dimensions:
Baggage: (1000, 5)
Flight: (250, 6)
Assignment: (1500, 5)
Scan Events: (9000, 9)


In [2]:
# Join baggage assignments with flight details

expected_journey = baggage_assignment.merge(
    flight,
    on="flight_id",
    how="left",
    validate="many_to_one"
)

# Sort flights in the correct journey order
expected_journey = expected_journey.sort_values(
    by=["bag_id", "segment_sequence"]
).reset_index(drop=True)

# Display expected journey columns
display(
    expected_journey[
        [
            "bag_id",
            "segment_sequence",
            "flight_number",
            "source",
            "destination",
            "departure_time",
            "arrival_time"
        ]
    ].head(10)
)

print("\nTotal expected flight segments:", len(expected_journey))
print("Unique baggage items:", expected_journey["bag_id"].nunique())

,bag_id,segment_sequence,flight_number,source,destination,departure_time,arrival_time
0,BAG00001,1,6E5268,DEL,LHR,2026-11-01 15:00:00+00:00,2026-11-01 18:00:00+00:00
1,BAG00002,1,EK7887,DXB,JFK,2026-11-02 06:00:00+00:00,2026-11-02 12:00:00+00:00
2,BAG00003,1,LH2805,DXB,JFK,2026-11-03 06:00:00+00:00,2026-11-03 12:00:00+00:00
3,BAG00004,1,LH4110,MAA,DEL,2026-11-04 06:00:00+00:00,2026-11-04 09:00:00+00:00
4,BAG00005,1,6E811,MAA,SIN,2026-11-05 06:00:00+00:00,2026-11-05 09:00:00+00:00
5,BAG00006,1,EK9395,BOM,DXB,2026-11-06 15:00:00+00:00,2026-11-06 18:00:00+00:00
6,BAG00007,1,LH3358,FRA,LHR,2026-11-07 06:00:00+00:00,2026-11-07 12:00:00+00:00
7,BAG00008,1,EK5413,MAA,DEL,2026-11-08 06:00:00+00:00,2026-11-08 09:00:00+00:00
8,BAG00009,1,6E1233,BOM,DXB,2026-11-09 15:00:00+00:00,2026-11-09 18:00:00+00:00
9,BAG00010,1,QR3532,SIN,LHR,2026-11-10 15:00:00+00:00,2026-11-10 21:00:00+00:00



Total expected flight segments: 1500
Unique baggage items: 1000


In [3]:
# Create a separate copy of actual scan events

actual_journey = baggage_scan_event.copy()

# Sort events chronologically for each baggage item
actual_journey = actual_journey.sort_values(
    by=["bag_id", "event_time", "event_id"]
).reset_index(drop=True)

# Display actual journey events
display(
    actual_journey[
        [
            "bag_id",
            "event_type",
            "airport_code",
            "flight_number",
            "event_time"
        ]
    ].head(15)
)

print("\nTotal actual scan events:", len(actual_journey))
print("Unique baggage items:", actual_journey["bag_id"].nunique())

,bag_id,event_type,airport_code,flight_number,event_time
0,BAG00001,CHECK_IN,DEL,NaN,2026-11-01 11:00:00+00:00
1,BAG00001,SECURITY_SCAN,DEL,NaN,2026-11-01 12:00:00+00:00
2,BAG00001,SORTED,DEL,6E5268,2026-11-01 13:00:00+00:00
3,BAG00001,LOADED,DEL,6E5268,2026-11-01 14:15:00+00:00
4,BAG00001,ARRIVED,LHR,6E5268,2026-11-01 18:00:00+00:00
5,BAG00001,UNLOADED,LHR,6E5268,2026-11-01 18:15:00+00:00
6,BAG00001,DELIVERED,LHR,NaN,2026-11-01 19:00:00+00:00
7,BAG00002,CHECK_IN,DXB,NaN,2026-11-02 02:00:00+00:00
8,BAG00002,SECURITY_SCAN,DXB,NaN,2026-11-02 03:00:00+00:00
9,BAG00002,SORTED,DXB,EK7887,2026-11-02 04:00:00+00:00



Total actual scan events: 9000
Unique baggage items: 1000


In [4]:
# Create actual airport sequence for each baggage item

actual_airport_sequence = (
    actual_journey
    .sort_values(["bag_id", "event_time"])
    .groupby("bag_id")["airport_code"]
    .apply(lambda airports: list(
        airports[airports.ne(airports.shift())]
    ))
    .reset_index(name="actual_airports")
)

# Display sample airport sequences
display(actual_airport_sequence.head(10))

print(
    "\nTotal baggage journeys:",
    len(actual_airport_sequence)
)

,bag_id,actual_airports
0,BAG00001,"[DEL, LHR]"
1,BAG00002,"[DXB, JFK]"
2,BAG00003,"[DXB, JFK]"
3,BAG00004,"[MAA, DEL]"
4,BAG00005,"[MAA, SIN]"
5,BAG00006,"[BOM, DXB]"
6,BAG00007,"[FRA, LHR]"
7,BAG00008,"[MAA, DEL]"
8,BAG00009,"[BOM, DXB]"
9,BAG00010,"[SIN, LHR]"



Total baggage journeys: 1000


In [5]:
# Build expected airport sequence for each baggage item

expected_airport_sequence = (
    expected_journey
    .sort_values(["bag_id", "segment_sequence"])
    .groupby("bag_id")
    .apply(
        lambda group: [
            group.iloc[0]["source"]
        ] + group["destination"].tolist(),
        include_groups=False
    )
    .reset_index(name="expected_airports")
)

# Display sample expected journeys
display(expected_airport_sequence.head(10))

print(
    "\nTotal expected baggage journeys:",
    len(expected_airport_sequence)
)

,bag_id,expected_airports
0,BAG00001,"[DEL, LHR]"
1,BAG00002,"[DXB, JFK]"
2,BAG00003,"[DXB, JFK]"
3,BAG00004,"[MAA, DEL]"
4,BAG00005,"[MAA, SIN]"
5,BAG00006,"[BOM, DXB]"
6,BAG00007,"[FRA, LHR]"
7,BAG00008,"[MAA, DEL]"
8,BAG00009,"[BOM, DXB]"
9,BAG00010,"[SIN, LHR]"



Total expected baggage journeys: 1000


In [6]:
# Merge expected and actual airport sequences

journey_comparison = expected_airport_sequence.merge(
    actual_airport_sequence,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

# Compare airport sequences
journey_comparison["journey_matches"] = (
    journey_comparison["expected_airports"] ==
    journey_comparison["actual_airports"]
)

# Display comparison results
display(journey_comparison.head(10))

print("\nTotal journeys:", len(journey_comparison))

print(
    "Matching journeys:",
    journey_comparison["journey_matches"].sum()
)

print(
    "Non-matching journeys:",
    (~journey_comparison["journey_matches"]).sum()
)

,bag_id,expected_airports,actual_airports,journey_matches
0,BAG00001,"[DEL, LHR]","[DEL, LHR]",True
1,BAG00002,"[DXB, JFK]","[DXB, JFK]",True
2,BAG00003,"[DXB, JFK]","[DXB, JFK]",True
3,BAG00004,"[MAA, DEL]","[MAA, DEL]",True
4,BAG00005,"[MAA, SIN]","[MAA, SIN]",True
5,BAG00006,"[BOM, DXB]","[BOM, DXB]",True
6,BAG00007,"[FRA, LHR]","[FRA, LHR]",True
7,BAG00008,"[MAA, DEL]","[MAA, DEL]",True
8,BAG00009,"[BOM, DXB]","[BOM, DXB]",True
9,BAG00010,"[SIN, LHR]","[SIN, LHR]",True



Total journeys: 1000
Matching journeys: 815
Non-matching journeys: 185


In [7]:
# Extract journeys where expected and actual airports differ

mismatched_journeys = journey_comparison[
    journey_comparison["journey_matches"] == False
].copy()

print(
    "Total mismatched journeys:",
    len(mismatched_journeys)
)

display(mismatched_journeys.head(15))

Total mismatched journeys: 185


,bag_id,expected_airports,actual_airports,journey_matches
500,BAG00501,"[BOM, DXB, JFK]","[DXB, BOM, JFK, BOM, JFK, BOM, DXB]",False
505,BAG00506,"[DEL, DOH, LHR]","[DOH, DEL, LHR, DEL, LHR, DEL, DOH]",False
508,BAG00509,"[DEL, DOH, LHR]","[DOH, DEL, LHR, DEL, LHR, DEL, DOH]",False
509,BAG00510,"[BLR, SIN, FRA]","[SIN, BLR, FRA, BLR, FRA, BLR, SIN]",False
510,BAG00511,"[BOM, DXB, JFK]","[DXB, BOM, JFK, BOM, JFK, BOM, DXB]",False
512,BAG00513,"[DEL, DOH, LHR]","[DOH, DEL, LHR, DEL, LHR, DEL, DOH]",False
513,BAG00514,"[DEL, DOH, LHR]","[DOH, DEL, LHR, DEL, LHR, DEL, DOH]",False
520,BAG00521,"[BLR, SIN, FRA]","[SIN, BLR, FRA, BLR, FRA, BLR, SIN]",False
521,BAG00522,"[BOM, DXB, JFK]","[DXB, BOM, JFK, BOM, JFK, BOM, DXB]",False
524,BAG00525,"[BOM, DXB, JFK]","[DXB, BOM, JFK, BOM, JFK, BOM, DXB]",False


In [8]:
# Select one mismatched baggage item
sample_bag_id = "BAG00501"

# Retrieve its complete actual scan history
bag_event_history = actual_journey[
    actual_journey["bag_id"] == sample_bag_id
].copy()

# Display chronological event history
display(
    bag_event_history[
        [
            "event_id",
            "event_type",
            "airport_code",
            "flight_number",
            "event_time",
            "received_at"
        ]
    ]
)

# Display expected flight segments
print("\nExpected flight segments:")

display(
    expected_journey[
        expected_journey["bag_id"] == sample_bag_id
    ][
        [
            "segment_sequence",
            "flight_number",
            "source",
            "destination",
            "departure_time",
            "arrival_time"
        ]
    ]
)

,event_id,event_type,airport_code,flight_number,event_time,received_at
3500,EVT003508,LOADED,DXB,6E2160,2026-11-01 05:15:00+00:00,2026-11-01 05:16:00+00:00
3501,EVT003501,CHECK_IN,BOM,NaN,2026-11-01 11:00:00+00:00,2026-11-01 11:00:00+00:00
3502,EVT003502,SECURITY_SCAN,BOM,NaN,2026-11-01 12:00:00+00:00,2026-11-01 12:15:00+00:00
3503,EVT003509,ARRIVED,JFK,6E2160,2026-11-01 12:00:00+00:00,2026-11-01 12:01:00+00:00
3504,EVT003510,UNLOADED,JFK,6E2160,2026-11-01 12:15:00+00:00,2026-11-01 12:30:00+00:00
3505,EVT003503,SORTED,BOM,AI9913,2026-11-01 13:00:00+00:00,2026-11-01 13:00:00+00:00
3506,EVT003511,DELIVERED,JFK,NaN,2026-11-01 13:00:00+00:00,2026-11-01 13:00:00+00:00
3507,EVT003504,LOADED,BOM,AI9913,2026-11-01 14:15:00+00:00,2026-11-01 14:15:00+00:00
3508,EVT003505,ARRIVED,DXB,AI9913,2026-11-01 18:00:00+00:00,2026-11-01 18:00:00+00:00
3509,EVT003506,UNLOADED,DXB,AI9913,2026-11-01 18:15:00+00:00,2026-11-01 18:15:00+00:00



Expected flight segments:


,segment_sequence,flight_number,source,destination,departure_time,arrival_time
500,1,AI9913,BOM,DXB,2026-11-01 15:00:00+00:00,2026-11-01 18:00:00+00:00
501,2,6E2160,DXB,JFK,2026-11-01 06:00:00+00:00,2026-11-01 12:00:00+00:00


In [9]:
# Extract the first and second flight segments

connecting_flights = expected_journey[
    expected_journey["segment_sequence"].isin([1, 2])
].copy()

# Separate first and second flight details
first_flight = connecting_flights[
    connecting_flights["segment_sequence"] == 1
][
    ["bag_id", "flight_number", "arrival_time"]
].rename(
    columns={
        "flight_number": "first_flight",
        "arrival_time": "first_arrival"
    }
)

second_flight = connecting_flights[
    connecting_flights["segment_sequence"] == 2
][
    ["bag_id", "flight_number", "departure_time"]
].rename(
    columns={
        "flight_number": "second_flight",
        "departure_time": "second_departure"
    }
)

# Combine first and second flight details
connection_analysis = first_flight.merge(
    second_flight,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

# Calculate connection gap in minutes
connection_analysis["connection_gap_minutes"] = (
    connection_analysis["second_departure"] -
    connection_analysis["first_arrival"]
).dt.total_seconds() / 60

# Flag impossible connections
connection_analysis["impossible_connection"] = (
    connection_analysis["connection_gap_minutes"] < 0
)

print("Total connecting journeys:", len(connection_analysis))

print(
    "Impossible connections:",
    connection_analysis["impossible_connection"].sum()
)

display(
    connection_analysis.head(10)
)

Total connecting journeys: 500
Impossible connections: 185


,bag_id,first_flight,first_arrival,second_flight,second_departure,connection_gap_minutes,impossible_connection
0,BAG00501,AI9913,2026-11-01 18:00:00+00:00,6E2160,2026-11-01 06:00:00+00:00,-720.0,True
1,BAG00502,AI4606,2026-11-02 09:00:00+00:00,EK9028,2026-11-02 15:00:00+00:00,360.0,False
2,BAG00503,QR6074,2026-11-03 09:00:00+00:00,SQ506,2026-11-03 15:00:00+00:00,360.0,False
3,BAG00504,EK2366,2026-11-04 09:00:00+00:00,LH288,2026-11-04 15:00:00+00:00,360.0,False
4,BAG00505,QR3541,2026-11-05 09:00:00+00:00,6E6845,2026-11-05 15:00:00+00:00,360.0,False
5,BAG00506,SQ9662,2026-11-06 18:00:00+00:00,LH9324,2026-11-06 06:00:00+00:00,-720.0,True
6,BAG00507,6E1896,2026-11-07 09:00:00+00:00,6E4471,2026-11-07 15:00:00+00:00,360.0,False
7,BAG00508,QR9273,2026-11-08 09:00:00+00:00,6E1871,2026-11-08 15:00:00+00:00,360.0,False
8,BAG00509,EK5247,2026-11-09 18:00:00+00:00,LH4445,2026-11-09 06:00:00+00:00,-720.0,True
9,BAG00510,AI1853,2026-11-10 18:00:00+00:00,AI158,2026-11-10 06:00:00+00:00,-720.0,True


In [10]:
# Combine airport comparison and connection analysis

baggage_investigation = journey_comparison.merge(
    connection_analysis[
        [
            "bag_id",
            "connection_gap_minutes",
            "impossible_connection"
        ]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Mark direct journeys separately
baggage_investigation["journey_type"] = (
    baggage_investigation["connection_gap_minutes"]
    .notna()
    .map({
        True: "Connecting",
        False: "Direct"
    })
)

# Display investigation summary
display(
    baggage_investigation.head(15)
)

print("\nTotal baggage:", len(baggage_investigation))

print(
    "Direct journeys:",
    (baggage_investigation["journey_type"] == "Direct").sum()
)

print(
    "Connecting journeys:",
    (baggage_investigation["journey_type"] == "Connecting").sum()
)

print(
    "Impossible connections:",
    baggage_investigation["impossible_connection"].sum()
)

,bag_id,expected_airports,actual_airports,journey_matches,connection_gap_minutes,impossible_connection,journey_type
0,BAG00001,"[DEL, LHR]","[DEL, LHR]",True,NaN,NaN,Direct
1,BAG00002,"[DXB, JFK]","[DXB, JFK]",True,NaN,NaN,Direct
2,BAG00003,"[DXB, JFK]","[DXB, JFK]",True,NaN,NaN,Direct
3,BAG00004,"[MAA, DEL]","[MAA, DEL]",True,NaN,NaN,Direct
4,BAG00005,"[MAA, SIN]","[MAA, SIN]",True,NaN,NaN,Direct
5,BAG00006,"[BOM, DXB]","[BOM, DXB]",True,NaN,NaN,Direct
6,BAG00007,"[FRA, LHR]","[FRA, LHR]",True,NaN,NaN,Direct
7,BAG00008,"[MAA, DEL]","[MAA, DEL]",True,NaN,NaN,Direct
8,BAG00009,"[BOM, DXB]","[BOM, DXB]",True,NaN,NaN,Direct
9,BAG00010,"[SIN, LHR]","[SIN, LHR]",True,NaN,NaN,Direct



Total baggage: 1000
Direct journeys: 500
Connecting journeys: 500
Impossible connections: 185


In [11]:
# Identify second-flight loading events

second_loading_events = actual_journey.merge(
    connection_analysis[["bag_id", "second_flight"]],
    on="bag_id",
    how="inner"
)

second_loading_events = second_loading_events[
    (second_loading_events["event_type"] == "LOADED") &
    (second_loading_events["flight_number"] ==
     second_loading_events["second_flight"])
]

second_loading_events = second_loading_events[
    ["bag_id", "event_time"]
].rename(
    columns={"event_time": "second_loading_time"}
)

# Identify transfer events
transfer_times = (
    actual_journey[
        actual_journey["event_type"] == "TRANSFER"
    ][["bag_id", "event_time"]]
    .rename(columns={"event_time": "transfer_time"})
)

# Compare loading and transfer times
handoff_analysis = second_loading_events.merge(
    transfer_times,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

handoff_analysis["loading_after_transfer"] = (
    handoff_analysis["second_loading_time"] >
    handoff_analysis["transfer_time"]
)

print("Connecting baggage checked:", len(handoff_analysis))

print(
    "Second flight loaded after transfer:",
    handoff_analysis["loading_after_transfer"].sum()
)

print(
    "Second flight loaded before transfer:",
    (~handoff_analysis["loading_after_transfer"]).sum()
)

display(handoff_analysis.head(10))

Connecting baggage checked: 500
Second flight loaded after transfer: 315
Second flight loaded before transfer: 185


,bag_id,second_loading_time,transfer_time,loading_after_transfer
0,BAG00501,2026-11-01 05:15:00+00:00,2026-11-01 18:45:00+00:00,False
1,BAG00502,2026-11-02 14:15:00+00:00,2026-11-02 09:45:00+00:00,True
2,BAG00503,2026-11-03 14:15:00+00:00,2026-11-03 09:45:00+00:00,True
3,BAG00504,2026-11-04 14:15:00+00:00,2026-11-04 09:45:00+00:00,True
4,BAG00505,2026-11-05 14:15:00+00:00,2026-11-05 09:45:00+00:00,True
5,BAG00506,2026-11-06 05:15:00+00:00,2026-11-06 18:45:00+00:00,False
6,BAG00507,2026-11-07 14:15:00+00:00,2026-11-07 09:45:00+00:00,True
7,BAG00508,2026-11-08 14:15:00+00:00,2026-11-08 09:45:00+00:00,True
8,BAG00509,2026-11-09 05:15:00+00:00,2026-11-09 18:45:00+00:00,False
9,BAG00510,2026-11-10 05:15:00+00:00,2026-11-10 18:45:00+00:00,False


In [12]:
# Merge handoff analysis into the investigation dataset

baggage_investigation = baggage_investigation.merge(
    handoff_analysis[
        [
            "bag_id",
            "second_loading_time",
            "transfer_time",
            "loading_after_transfer"
        ]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Add an investigation classification
baggage_investigation["investigation_status"] = np.select(
    [
        baggage_investigation["impossible_connection"].eq(True),
        baggage_investigation["journey_matches"].eq(False),
        baggage_investigation["journey_matches"].eq(True)
    ],
    [
        "TIMING_INCONSISTENCY",
        "AIRPORT_SEQUENCE_MISMATCH",
        "NO_AIRPORT_MISMATCH"
    ],
    default="REQUIRES_REVIEW"
)

# Display investigation results
display(
    baggage_investigation[
        [
            "bag_id",
            "expected_airports",
            "actual_airports",
            "journey_matches",
            "connection_gap_minutes",
            "impossible_connection",
            "loading_after_transfer",
            "investigation_status"
        ]
    ].head(15)
)

print("\nInvestigation status summary:")

display(
    baggage_investigation["investigation_status"]
    .value_counts()
)

,bag_id,expected_airports,actual_airports,journey_matches,connection_gap_minutes,impossible_connection,loading_after_transfer,investigation_status
0,BAG00001,"[DEL, LHR]","[DEL, LHR]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
1,BAG00002,"[DXB, JFK]","[DXB, JFK]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
2,BAG00003,"[DXB, JFK]","[DXB, JFK]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
3,BAG00004,"[MAA, DEL]","[MAA, DEL]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
4,BAG00005,"[MAA, SIN]","[MAA, SIN]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
5,BAG00006,"[BOM, DXB]","[BOM, DXB]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
6,BAG00007,"[FRA, LHR]","[FRA, LHR]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
7,BAG00008,"[MAA, DEL]","[MAA, DEL]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
8,BAG00009,"[BOM, DXB]","[BOM, DXB]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH
9,BAG00010,"[SIN, LHR]","[SIN, LHR]",True,NaN,NaN,NaN,NO_AIRPORT_MISMATCH



Investigation status summary:


investigation_status
NO_AIRPORT_MISMATCH     815
TIMING_INCONSISTENCY    185
Name: count, dtype: int64

In [13]:
# Summarize investigation results by journey type

investigation_summary = (
    baggage_investigation
    .groupby(["journey_type", "investigation_status"])
    .size()
    .reset_index(name="Baggage Count")
)

display(investigation_summary)

print("\nJourney type totals:")

display(
    baggage_investigation["journey_type"]
    .value_counts()
)

,journey_type,investigation_status,Baggage Count
0,Connecting,NO_AIRPORT_MISMATCH,315
1,Connecting,TIMING_INCONSISTENCY,185
2,Direct,NO_AIRPORT_MISMATCH,500



Journey type totals:


journey_type
Direct        500
Connecting    500
Name: count, dtype: int64

In [14]:
# Save consolidated baggage investigation results

output_path = processed_path / "baggage_investigation.csv"

baggage_investigation.to_csv(
    output_path,
    index=False
)

print("Investigation dataset saved successfully!")
print("File:", output_path)
print("Rows:", len(baggage_investigation))
print("Columns:", len(baggage_investigation.columns))

Investigation dataset saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\baggage_investigation.csv
Rows: 1000
Columns: 11


In [15]:
# Reload the saved investigation dataset
baggage_investigation_verified = pd.read_csv(output_path)

# Verify the saved output
print("Original rows:", len(baggage_investigation))
print("Reloaded rows:", len(baggage_investigation_verified))

print("\nColumn names match:",
      list(baggage_investigation.columns) ==
      list(baggage_investigation_verified.columns))

print("\nUnique baggage IDs:",
      baggage_investigation_verified["bag_id"].nunique())

print("\nInvestigation status after reload:")
display(
    baggage_investigation_verified["investigation_status"]
    .value_counts()
)

# Check for missing baggage IDs
print("\nMissing baggage IDs:",
      baggage_investigation_verified["bag_id"].isna().sum())

Original rows: 1000
Reloaded rows: 1000

Column names match: True

Unique baggage IDs: 1000

Investigation status after reload:


investigation_status
NO_AIRPORT_MISMATCH     815
TIMING_INCONSISTENCY    185
Name: count, dtype: int64


Missing baggage IDs: 0
